# Text -> SVG  (SDXL-Turbo + VTracer)

Darmowy Colab, T4. Ustaw **Runtime -> Change runtime type -> T4 GPU**.

1. Odpal komorke 1 (instalacja, ~2 min).
2. Odpal komorke 2 (model + interfejs Gradio).
3. Kliknij link `Running on public URL` albo uzyj UI pod komorka.

Model generuje raster, VTracer trasuje go do SVG. Pobierz plik `.svg` z panelu po prawej.

In [ ]:
!pip install -q diffusers transformers accelerate vtracer gradio
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, uuid, torch, vtracer
import gradio as gr
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")

os.makedirs("out", exist_ok=True)

STYLE = ("flat vector illustration, minimal, bold solid colors, "
         "no gradients, no texture, no shadow, clean shapes, white background")


def generate(prompt, steps, seed, filter_speckle, color_precision,
             layer_difference, path_precision, mode):
    if not prompt.strip():
        raise gr.Error("Prompt jest pusty.")

    seed = int(seed)
    gen = torch.Generator("cuda").manual_seed(seed) if seed >= 0 else None

    img = pipe(
        prompt=f"{prompt}, {STYLE}",
        num_inference_steps=int(steps),
        guidance_scale=0.0,
        height=512, width=512,
        generator=gen,
    ).images[0]

    uid = uuid.uuid4().hex[:8]
    png, svg = f"out/{uid}.png", f"out/{uid}.svg"
    img.save(png)

    vtracer.convert_image_to_svg_py(
        png, svg,
        colormode="color",
        mode=mode,
        hierarchical="stacked",
        filter_speckle=int(filter_speckle),
        color_precision=int(color_precision),
        layer_difference=int(layer_difference),
        path_precision=int(path_precision),
    )

    with open(svg, encoding="utf-8") as f:
        src = f.read()

    kb = len(src.encode()) / 1024
    paths = src.count("<path")
    warn = "  **Duzo sciezek** - podnies filter_speckle / zejdz z color_precision." if paths > 400 else ""
    info = f"`{svg}` - {kb:.1f} KB, {paths} sciezek.{warn}"
    return img, f'<div style="background:#fff;padding:8px;overflow:auto">{src}</div>', svg, info


with gr.Blocks(title="Text to SVG") as demo:
    gr.Markdown("## Text -> SVG &nbsp;·&nbsp; SDXL-Turbo + VTracer")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", lines=3,
                                value="mountain and sun logo")
            steps = gr.Slider(1, 8, 4, step=1, label="Kroki (turbo: 1-4 wystarcza)")
            seed = gr.Number(-1, label="Seed (-1 = losowy)", precision=0)

            gr.Markdown("### Trasowanie (VTracer)")
            mode = gr.Radio(["spline", "polygon", "pixel"], value="spline",
                            label="Krzywe")
            filter_speckle = gr.Slider(0, 32, 8, step=1,
                                       label="filter_speckle - kasuje plamki")
            color_precision = gr.Slider(1, 8, 5, step=1,
                                        label="color_precision - liczba kolorow")
            layer_difference = gr.Slider(0, 64, 24, step=1,
                                         label="layer_difference - scala warstwy")
            path_precision = gr.Slider(0, 8, 4, step=1,
                                       label="path_precision - miejsca dziesietne")
            btn = gr.Button("Generuj", variant="primary")

        with gr.Column(scale=2):
            out_png = gr.Image(label="Raster (podglad)", type="pil")
            out_svg = gr.HTML(label="SVG")
            out_file = gr.File(label="Pobierz SVG")
            out_info = gr.Markdown()

    btn.click(
        generate,
        [prompt, steps, seed, filter_speckle, color_precision,
         layer_difference, path_precision, mode],
        [out_png, out_svg, out_file, out_info],
    )

demo.launch(share=True)